In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("C:\\Users\\bhavy\\OneDrive\\Desktop\\Projects\\Bank_Personal_Loan\\Bank_Personal_Loan_Modelling.xlsx",sheet_name=1)

In [3]:
data = df.copy()
data = data.drop(['ID','ZIP Code'],axis=1)

In [4]:
data = data[['Age', 'Experience', 'Income', 'Family', 'CCAvg',
       'Education', 'Mortgage', 'Securities Account',
       'CD Account', 'Online', 'CreditCard','Personal Loan']]

In [5]:
data = data[data['Experience'] >= 0]

In [6]:
data.shape

(4948, 12)

In [7]:
data['CCToIncomeRatio'] = data['CCAvg'] / (data['Income']/12)

In [8]:
data = data[['Age','Experience', 'Income', 'Family', 'CCAvg',
       'Education', 'Mortgage','Securities Account', 'CD Account', 'Online',
       'CreditCard','CCToIncomeRatio','Personal Loan']]

In [9]:
data = data[['Age','Income','CD Account','Mortgage','Education','CCAvg','CCToIncomeRatio','Personal Loan']]

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
X = data.iloc[:,:-1]
y = data.iloc[:,-1]

In [12]:
X.shape

(4948, 7)

In [13]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [14]:
print(X_train.shape)
X_test.shape

(3711, 7)


(1237, 7)

In [15]:
y_train.value_counts()

Personal Loan
0    3336
1     375
Name: count, dtype: int64

In [16]:
from imblearn.over_sampling import SMOTENC

smotenc = SMOTENC(categorical_features=[2,4],random_state=42)

In [17]:
X_train_balanced, y_train_balanced = smotenc.fit_resample(X_train, y_train)

In [18]:
y_train_balanced.value_counts()

Personal Loan
0    3336
1    3336
Name: count, dtype: int64

In [19]:
from sklearn.preprocessing import StandardScaler,RobustScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [20]:
Transformer = ColumnTransformer([
    ('StandardScaling',StandardScaler(),['Age','CCToIncomeRatio']),
    ('RobustScaling',RobustScaler(),['Income','Mortgage','CCAvg']),
    ('OHE',OneHotEncoder(drop='first',sparse_output=False),['CD Account'])
],remainder='passthrough')

In [21]:
X_train_transformed = Transformer.fit_transform(X_train_balanced)

In [22]:
X_test_transformed = Transformer.transform(X_test)

In [23]:
import optuna

In [24]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [25]:
from sklearn.model_selection import cross_val_score

In [26]:
def objective(trial):

    classifier_model = trial.suggest_categorical('classifier',['SVC', 'Logistic','RFC','GB'])

    if classifier_model =='SVC':
        c = trial.suggest_float('svc_C',0.1,100,log=True)
        degree = trial.suggest_int('degree',1,4)
        kernel = trial.suggest_categorical('kernel',['linear', 'poly', 'rbf', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma',['scale', 'auto'])

        model = SVC(C=c,degree=degree,kernel=kernel,gamma=gamma,random_state=42)

    elif classifier_model=='Logistic':
        
        C = trial.suggest_float('logistic_C',0.1,100,log=True)
        class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])
        solver = trial.suggest_categorical('solver',['lbfgs','liblinear'])

        model = LogisticRegression(C=C,class_weight=class_weight,solver=solver,max_iter=1000)

    elif classifier_model=='RFC':
        n_estimators = trial.suggest_int('estimators',20,150,step=2)
        criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
        max_depth = trial.suggest_int('max_depth',3,7)
        min_samples_split = trial.suggest_int('min_samples_split',2,10,step=2)
        min_samples_leaf = trial.suggest_int('min_samples_leaf',2,16,step=2)
        max_features = trial.suggest_categorical('max_features',['sqrt','log2'])
        bootstrap = trial.suggest_categorical('bootstrap',[True,False])

        model = RandomForestClassifier(n_estimators=n_estimators,criterion=criterion,max_depth=max_depth,min_samples_split=min_samples_split,
                                      min_samples_leaf=min_samples_leaf,max_features=max_features,bootstrap=bootstrap,random_state=42)

    elif classifier_model == 'GB':
        n_estimators=trial.suggest_int('n_estimators',50,150,step=2)
        learning_rate=trial.suggest_float('learning_rate',0.01,0.3,log=True)
        min_samples_split = trial.suggest_int('min_samples_split',2,10,step=2)
        min_samples_leaf = trial.suggest_int('min_samples_leaf',2,16,step=2)

        model = GradientBoostingClassifier(n_estimators=n_estimators,learning_rate=learning_rate,
                                           min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,random_state=42)

    
    score = cross_val_score(model, X_train_transformed, y_train_balanced, cv=3, scoring='recall').mean()
    return score  # Return the accuracy score for Optuna to maximize

In [27]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=125)  # Run 50 trials to find the best hyperparameters

[I 2026-07-19 19:43:18,289] A new study created in memory with name: no-name-eeb58784-1b2e-4567-bed0-dfce2cc0c1d5
[I 2026-07-19 19:43:18,325] Trial 0 finished with value: 0.8944844124700239 and parameters: {'classifier': 'Logistic', 'logistic_C': 1.9618234983891285, 'class_weight': None, 'solver': 'liblinear'}. Best is trial 0 with value: 0.8944844124700239.
[I 2026-07-19 19:43:18,402] Trial 1 finished with value: 0.8983812949640289 and parameters: {'classifier': 'Logistic', 'logistic_C': 0.13935518591246635, 'class_weight': None, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.8983812949640289.
[I 2026-07-19 19:43:20,367] Trial 2 finished with value: 0.9127697841726619 and parameters: {'classifier': 'SVC', 'svc_C': 17.405620331447352, 'degree': 2, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 2 with value: 0.9127697841726619.
[I 2026-07-19 19:43:20,852] Trial 3 finished with value: 0.9775179856115108 and parameters: {'classifier': 'RFC', 'estimators': 50, 'criterion': 'entrop

In [28]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.9901079136690648
Best hyperparameters: {'classifier': 'RFC', 'estimators': 24, 'criterion': 'gini', 'max_depth': 4, 'min_samples_split': 4, 'min_samples_leaf': 16, 'max_features': 'log2', 'bootstrap': False}


In [29]:
RFC_model = RandomForestClassifier(n_estimators=24,criterion='gini',max_depth=4,min_samples_split=4,min_samples_leaf=16,
                                   max_features='log2',bootstrap=False,random_state=42)

In [30]:
# X_train and y_train & X_test and y_test are raw test_train_split
# X_train_balanced and y_train_balanced where made by SMOTENC on X_train, y_train
# X_train_transformed and X_test_transformed were formed after using the ColumnTransformer on X_train_balanced and X_test

In [31]:
from imblearn.pipeline import Pipeline    # Not using sklearn's pipeline, because it does not work with SMOTENC

In [33]:
final_pipe = Pipeline(
    [('Imbalance Handling',smotenc),
     ('Preprocessing',Transformer),
     ('Final RFC Model',RFC_model)
    ]
)

In [34]:
final_pipe.fit(X_train,y_train)

C:\Users\bhavy\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('Imbalance Handling',
                 SMOTENC(categorical_features=[2, 4], random_state=42)),
                ('Preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('StandardScaling',
                                                  StandardScaler(),
                                                  ['Age', 'CCToIncomeRatio']),
                                                 ('RobustScaling',
                                                  RobustScaler(),
                                                  ['Income', 'Mortgage',
                                                   'CCAvg']),
                                                 ('OHE',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['CD Account'])])),
                ('Final RFC Model',
                 RandomForestClassifier(bootstrap=False, max_depth=4,
                                        max_features='log2',
                                        min_samples_leaf=16,
                                        min_samples_split=4, n_estimators=24,
                                        random_state=42))])

In [36]:
y_pred = final_pipe.predict(X_test)

In [38]:
from sklearn.metrics import confusion_matrix, classification_report

In [40]:
print('Confusion Matrix for RFC')
print(confusion_matrix(y_test,y_pred))

Confusion Matrix for RFC
[[976 156]
 [  3 102]]


In [41]:
print('Classification report for RFC')
print(classification_report(y_test,y_pred))

Classification report for RFC
              precision    recall  f1-score   support

           0       1.00      0.86      0.92      1132
           1       0.40      0.97      0.56       105

    accuracy                           0.87      1237
   macro avg       0.70      0.92      0.74      1237
weighted avg       0.95      0.87      0.89      1237

